# Chapter 4 &mdash; The Pumping Lemma: Statement and Pigeonhole

**Concept 14 of the Chapter 4 decomposition:** *The Pumping Lemma for Regular Languages: Statement, Proof Sketch, and Pigeonhole*

If $L$ is regular then long strings split as $xyz$ with $|xy|\le N$, $y\ne\varepsilon$, and $xy^iz\in L$ for all $i$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Pumping-Lemma-Statement/Concept-Pumping-Lemma-Statement.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


> **IF** $L$ is regular, **THEN** there is $N$ such that for any $w\in L$ with
> $|w|\ge N$, we can write $w = xyz$ where $y$ is non-empty, $|xy| \le N$, and
> **for all $i \ge 0$, $xy^iz \in L$**.

**Why a repeat must occur:** the **pigeonhole principle**. With $M \ge N$ transitions
there must be a repeated state &mdash; states are like duck digits and transitions the
webs between them, so an $N$-state DFA admits at most a journey of length $N-1$
without repeating.

**Practical advice: pick $y$ first.** Then $x$ is whatever precedes it and $z$ is the
rest.

## 2. Definitions

### Pigeonhole, demonstrated

In [ ]:
def must_repeat(D, s):
    """Any run of length >= |Q| revisits a state."""
    q, seen = D["q0"], [D["q0"]]
    for ch in s:
        q = step_dfa(D, q, ch); seen.append(q)
    return len(seen) > len(set(seen)), len(seen), len(set(seen))

### The lemma as a checkable predicate

In [ ]:
D = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
N = len(D["Q"])

def pumping_splits(w, N):
    """All splits with y non-empty and |xy| <= N."""
    return [(w[:i], w[i:j], w[j:])
            for i in range(N+1) for j in range(i+1, min(N, len(w))+1)]

## 3. Tests

Pigeonhole: a long enough run cannot avoid a repeat.

In [ ]:
for w in ['01', '010', '0100', '010010']:
    rep, visits, distinct = must_repeat(D, w)
    print("|w|=%d  visits=%d distinct=%d  repeated? %s" % (len(w), visits, distinct, rep))
assert must_repeat(D, '0100')[0]
print("\n|Q| = %d, so any run of %d+ states must reuse one." % (N, N+1))

For a string in the language, **some** split pumps &mdash; the lemma promises at least one.

In [ ]:
w = '0100'
good = [(x,y,z) for (x,y,z) in pumping_splits(w, N)
        if y and all(accepts_dfa(D, x + y*i + z) for i in range(6))]
print("w =", w, " splits satisfying the lemma:")
for x,y,z in good: print("   x=%-4r y=%-4r z=%-4r" % (x,y,z))
assert good, "the lemma guarantees at least one pumping split"

Note the direction: the lemma says **some** split works, not **every** one.

In [ ]:
bad = [(x,y,z) for (x,y,z) in pumping_splits(w, N)
       if y and not all(accepts_dfa(D, x + y*i + z) for i in range(6))]
print("splits that do NOT pump :", len(bad))
print("\nThat is fine -- the lemma is existential over splits when PROVING regularity,")
print("which is exactly why refuting it needs ALL splits (Concept 17).")

## 4. Exercises


1. State the pigeonhole principle in one sentence, then apply it to a 5-state DFA.
2. Why must $|xy| \le N$? Which choice in the proof forces it?
3. Pick $y$ first for the string `010010`. Do $x$ and $z$ follow automatically?

In [ ]:
# Your work for the exercises above.